# Leakage-aware re-evaluation on CICDDoS2019 (Google Colab)

Runs a **day-wise** evaluation (train: 12 January capture, test: 11 March capture) with deduplication, train-only scaling/PCA, several seeds, calibration (Brier, ECE) and an inference-timing protocol. A **random pooled 70:30** comparison is optional so the optimism of random splits can be quantified.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Run cells top to bottom. After the *loading* cell, **stop and check the table** (which day each file was assigned to). Do not trust the day assignment until you have looked at it.

**Honest notes (copy into the paper's methods):** the model definitions below are *reconstructions* from the recorded unit counts (hybrid: 500 and 300 DNN-LSTM units, 200 DNN-GRU units, ReLU, softmax, batch 32, up to 10 epochs); the baseline architectures, optimizer (Adam 1e-3) and loss (sparse categorical cross-entropy) are choices made here, not recovered settings. Results are a **new experiment**, not a reproduction of earlier numbers.

In [ ]:
import os, sys, json, time, glob, re, shutil, platform, subprocess, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
    except Exception as e:
        return str(e)

print('TensorFlow', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
print(sh('nvidia-smi -L'))

TensorFlow 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU 0: Tesla T4 (UUID: GPU-74685854-464e-1798-0c6a-a1610cdfef68)


In [ ]:
import zipfile, os
os.makedirs('/content/data', exist_ok=True)
with zipfile.ZipFile('/content/archive.zip') as z:
    z.extractall('/content/data')
print(sh('find /content/data -type f | head -60; du -sh /content/data'))

/content/data/LDAP-training.parquet
/content/data/UDP-training.parquet
/content/data/MSSQL-training.parquet
/content/data/UDPLag-training.parquet
/content/data/MSSQL-testing.parquet
/content/data/TFTP-testing.parquet
/content/data/UDP-testing.parquet
/content/data/LDAP-testing.parquet
/content/data/NTP-testing.parquet
/content/data/SNMP-testing.parquet
/content/data/NetBIOS-testing.parquet
/content/data/Syn-testing.parquet
/content/data/Portmap-training.parquet
/content/data/Syn-training.parquet
/content/data/DNS-testing.parquet
/content/data/NetBIOS-training.parquet
/content/data/UDPLag-testing.parquet
34M	/content/data


In [ ]:
CFG = dict(
    DATA_DIR='/content/data',
    PER_CLASS_CAP=10000,        # max rows kept per class per file chunk and per file (bounds RAM and time)
    SEEDS=[0, 1, 2],            # raise to 5 if time allows
    DROP_PORTS=True,            # ports are campaign-specific identifiers; set False to keep them
    USE_PCA=True, PCA_COMPONENTS=30,
    EPOCHS=10, BATCH=32, VAL_FRAC=0.1,
    TIMING_REPEATS=500, TIMING_BATCHES=[1, 256], TIMING_WARMUP=50,
    RUN_RANDOM_POOLED_COMPARISON=True,
    MODELS=['hybrid', 'dnn_gru', 'blstm'],
    FILE_DAY_OVERRIDE={},       # e.g. {'/content/data/xyz.parquet': 'train'} if a file cannot be assigned automatically
    OUT_DIR='/content/results',
)
os.makedirs(CFG['OUT_DIR'], exist_ok=True)

In [ ]:
ID_COLS = {'flow id', 'source ip', 'destination ip', 'src ip', 'dst ip', 'timestamp', 'unnamed: 0',
           'simillarhttp', 'similarhttp', 'flowid', 'srcip', 'dstip'}
PORT_COLS = {'source port', 'destination port', 'src port', 'dst port'}
META = ['label_raw', 'label', '_row', '_src', '_day', '_h']

def day_of(path):
    if path in CFG['FILE_DAY_OVERRIDE']:
        return CFG['FILE_DAY_OVERRIDE'][path]
    p = path.lower()
    if re.search(r'train|01-12|0112|01_12', p):
        return 'train'
    if re.search(r'test|03-11|0311|03_11', p):
        return 'test'
    return 'unknown'

def norm_label(s):
    s = str(s).strip().lower()
    s = re.sub(r'^drdos[_\- ]?', '', s)
    return re.sub(r'[^a-z0-9]', '', s)

def iter_chunks(f):
    off = 0
    if f.endswith('.parquet'):
        import pyarrow.parquet as pq
        pf = pq.ParquetFile(f)
        gen = (b.to_pandas() for b in pf.iter_batches(batch_size=500000))
    else:
        gen = pd.read_csv(f, chunksize=500000, low_memory=False)
    for t in gen:
        t.columns = [str(c).strip() for c in t.columns]
        t['_row'] = np.arange(off, off + len(t))
        off += len(t)
        yield t

def cap_per_label(t, cap, rng):
    sel = []
    for lab, g in t.groupby('label'):
        idx = g.index.to_numpy()
        if len(idx) > cap:
            idx = rng.choice(idx, cap, replace=False)
        sel.append(idx)
    return t.loc[np.concatenate(sel)] if sel else t.iloc[:0]

def load_all():
    files_ = sorted(glob.glob(os.path.join(CFG['DATA_DIR'], '**', '*.parquet'), recursive=True) +
                    glob.glob(os.path.join(CFG['DATA_DIR'], '**', '*.csv'), recursive=True))
    rng = np.random.default_rng(12345)
    parts = []
    for f in files_:
        kept = []
        for t in iter_chunks(f):
            lc = next((c for c in t.columns if c.lower() == 'label'), None)
            if lc is None:
                raise ValueError('No Label column in %s: %s' % (f, list(t.columns)[:10]))
            t = t.rename(columns={lc: 'label_raw'})
            t['label'] = t['label_raw'].map(norm_label)
            kept.append(cap_per_label(t, CFG['PER_CLASS_CAP'], rng))
        if not kept:
            continue
        d = pd.concat(kept, ignore_index=True)
        d = cap_per_label(d, CFG['PER_CLASS_CAP'], rng).reset_index(drop=True)
        d['_src'] = os.path.relpath(f, CFG['DATA_DIR'])
        d['_day'] = day_of(f)
        parts.append(d)
    return pd.concat(parts, ignore_index=True)

In [ ]:
raw = load_all()
print('Rows loaded (capped):', len(raw))
print(raw.groupby('_day')['label'].value_counts().unstack(0).fillna(0).astype(int))
print(raw.groupby(['_src', '_day']).size())
assert 'unknown' not in set(raw['_day']), 'Some files were not assigned to train/test. Set CFG["FILE_DAY_OVERRIDE"].'
# >>> STOP: check that the day assignment matches the dataset (12 Jan = training day, 11 Mar = testing day).

Rows loaded (capped): 159316
_day      test  train
label                
benign   35182  29393
dns       3669      0
ldap      1440   1906
mssql     6212   8523
netbios    598    644
ntp      10000      0
portmap      0    685
snmp      2717      0
syn        533  15538
tftp     10000      0
udp      10000  13298
udplag    8872     55
webddos     51      0
_src                      _day 
DNS-testing.parquet       test      6703
LDAP-testing.parquet      test      2831
LDAP-training.parquet     train     6715
MSSQL-testing.parquet     test      8083
MSSQL-training.parquet    train    10974
NTP-testing.parquet       test     20000
NetBIOS-testing.parquet   test      2225
NetBIOS-training.parquet  train     1631
Portmap-training.parquet  train     5105
SNMP-testing.parquet      test      4018
Syn-testing.parquet       test       907
Syn-training.parquet      train    20000
TFTP-testing.parquet      test     20000
UDP-testing.parquet       test     12042
UDP-training.parquet      train    

In [ ]:
def clean(df):
    log = {'rows_in': int(len(df))}
    drop = [c for c in df.columns if c.lower() in ID_COLS or (CFG['DROP_PORTS'] and c.lower() in PORT_COLS)]
    feat = [c for c in df.columns if c not in META and c not in drop and pd.api.types.is_numeric_dtype(df[c])]
    X = df[feat].replace([np.inf, -np.inf], np.nan)
    ok = X.notna().all(axis=1)
    log['rows_removed_nan_or_inf'] = int((~ok).sum())
    df = df.loc[ok].copy()
    df[feat] = X.loc[ok].astype('float32')
    tr = df[df['_day'] == 'train']
    const = [c for c in feat if tr[c].nunique() <= 1]
    feat = [c for c in feat if c not in const]
    log['constant_features_dropped_using_train_day'] = const
    df['_h'] = pd.util.hash_pandas_object(df[feat], index=False).values
    log['hash_groups_with_conflicting_labels'] = int((df.groupby('_h')['label'].nunique() > 1).sum())
    n0 = len(df)
    df = df.sort_values('_day', ascending=False, kind='stable').drop_duplicates('_h', keep='first')
    log['rows_removed_exact_duplicates_incl_cross_day'] = int(n0 - len(df))
    log['rows_out'] = int(len(df))
    log['n_features'] = len(feat)
    return df.reset_index(drop=True), feat, log

df, FEAT, CLEAN_LOG = clean(raw)
print(json.dumps(CLEAN_LOG, indent=1))
print(df.groupby('_day')['label'].value_counts().unstack(0).fillna(0).astype(int))
assert 'benign' in set(df['label']), 'No benign class found: check label names'
train_df = df[df['_day'] == 'train'].reset_index(drop=True)
test_df = df[df['_day'] == 'test'].reset_index(drop=True)
print('train classes:', sorted(train_df['label'].unique()))
print('test classes :', sorted(test_df['label'].unique()))
print('classes only in test (unseen families):', sorted(set(test_df['label']) - set(train_df['label'])))

{
 "rows_in": 159316,
 "rows_removed_nan_or_inf": 0,
 "constant_features_dropped_using_train_day": [
  "Bwd PSH Flags",
  "Fwd URG Flags",
  "Bwd URG Flags",
  "FIN Flag Count",
  "PSH Flag Count",
  "ECE Flag Count",
  "Fwd Avg Bytes/Bulk",
  "Fwd Avg Packets/Bulk",
  "Fwd Avg Bulk Rate",
  "Bwd Avg Bytes/Bulk",
  "Bwd Avg Packets/Bulk",
  "Bwd Avg Bulk Rate"
 ],
 "hash_groups_with_conflicting_labels": 2487,
 "rows_removed_exact_duplicates_incl_cross_day": 10230,
 "rows_out": 149086,
 "n_features": 65
}
_day      test  train
label                
benign   34002  28444
dns       2837      0
ldap       683   1881
mssql     3335   8364
netbios    156    474
ntp       9997      0
portmap      0    437
snmp      1992      0
syn        533  15153
tftp      9989      0
udp       9362  12915
udplag    8445     37
webddos     50      0
train classes: ['benign', 'ldap', 'mssql', 'netbios', 'portmap', 'syn', 'udp', 'udplag']
test classes : ['benign', 'dns', 'ldap', 'mssql', 'netbios', 'ntp', 'sn

In [ ]:
class Pipe:
    def fit(self, X):
        self.sc = MinMaxScaler().fit(X)
        Z = self.sc.transform(X)
        self.pca = None
        if CFG['USE_PCA']:
            self.pca = PCA(n_components=min(CFG['PCA_COMPONENTS'], Z.shape[1]), random_state=0).fit(Z)
        return self
    def transform(self, X):
        Z = self.sc.transform(X)
        if self.pca is not None:
            Z = self.pca.transform(Z)
        return Z.astype('float32')

def ece_score(conf, correct, bins=15):
    edges = np.linspace(0, 1, bins + 1)
    e, n = 0.0, len(conf)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & (conf <= hi) if lo == 0 else (conf > lo) & (conf <= hi)
        if m.any():
            e += m.sum() / n * abs(correct[m].mean() - conf[m].mean())
    return float(e)

def evaluate(y_names, proba, classes):
    classes = list(classes)
    ben = classes.index('benign')
    pred = np.array(classes)[proba.argmax(1)]
    out = {}
    yb = (y_names != 'benign').astype(int)
    pb = (pred != 'benign').astype(int)
    p, r, f, _ = precision_recall_fscore_support(yb, pb, average='binary', zero_division=0)
    patt = 1 - proba[:, ben]
    out.update(bin_acc=float(accuracy_score(yb, pb)), bin_precision=float(p), bin_recall=float(r), bin_f1=float(f),
               bin_fpr=float(((pb == 1) & (yb == 0)).sum() / max((yb == 0).sum(), 1)),
               bin_brier=float(np.mean((patt - yb) ** 2)),
               bin_ece=ece_score(np.where(patt >= 0.5, patt, 1 - patt), (pb == yb).astype(float)),
               n_test=int(len(yb)), n_test_attack=int(yb.sum()))
    out['bin_auc'] = float(roc_auc_score(yb, patt)) if 0 < yb.sum() < len(yb) else float('nan')
    seen = np.isin(y_names, classes)
    ys, ps, Ps = y_names[seen], pred[seen], proba[seen]
    out['mc_n'] = int(seen.sum())
    out['mc_acc'] = float(accuracy_score(ys, ps))
    labs = [c for c in classes if c in set(ys)]
    p, r, f, s = precision_recall_fscore_support(ys, ps, labels=labs, average=None, zero_division=0)
    out['mc_macro_p'], out['mc_macro_r'], out['mc_macro_f1'] = float(np.mean(p)), float(np.mean(r)), float(np.mean(f))
    out['per_class'] = {l: dict(precision=float(a), recall=float(b), f1=float(c), support=int(d))
                        for l, a, b, c, d in zip(labs, p, r, f, s)}
    out['confusion'] = {'labels': classes, 'matrix': confusion_matrix(ys, ps, labels=classes).tolist()}
    yi = np.array([classes.index(v) for v in ys])
    out['mc_brier'] = float(np.mean(np.sum((Ps - np.eye(len(classes))[yi]) ** 2, axis=1)))
    out['mc_ece'] = ece_score(Ps.max(1), (Ps.argmax(1) == yi).astype(float))
    out['unseen_family_detected_as_attack'] = {u: float(np.mean(pred[y_names == u] != 'benign'))
                                                for u in sorted(set(y_names[~seen]))}
    return out

In [ ]:
from tensorflow.keras import layers as L

def build_hybrid(nf, nc):
    # RECONSTRUCTION from recorded unit counts: DNN-LSTM(500), DNN-LSTM(300), DNN-GRU(200), ReLU, softmax.
    i = tf.keras.Input((1, nf))
    x = L.Dense(500, activation='relu')(i); x = L.LSTM(500, return_sequences=True)(x)
    x = L.Dense(300, activation='relu')(x); x = L.LSTM(300, return_sequences=True)(x)
    x = L.Dense(200, activation='relu')(x); x = L.GRU(200)(x)
    return tf.keras.Model(i, L.Dense(nc, activation='softmax')(x), name='hybrid')

def build_dnn_gru(nf, nc):
    # Baseline defined here (original not recorded): Dense 500-300-200 (ReLU) then GRU(200).
    i = tf.keras.Input((1, nf))
    x = L.Dense(500, activation='relu')(i); x = L.Dense(300, activation='relu')(x)
    x = L.Dense(200, activation='relu')(x); x = L.GRU(200)(x)
    return tf.keras.Model(i, L.Dense(nc, activation='softmax')(x), name='dnn_gru')

def build_blstm(nf, nc):
    # Baseline defined here (original not recorded): Bidirectional LSTM(200) then Dense(100, ReLU).
    i = tf.keras.Input((1, nf))
    x = L.Bidirectional(L.LSTM(200))(i); x = L.Dense(100, activation='relu')(x)
    return tf.keras.Model(i, L.Dense(nc, activation='softmax')(x), name='blstm')

BUILDERS = {'hybrid': build_hybrid, 'dnn_gru': build_dnn_gru, 'blstm': build_blstm}

def fit_model(model, Xtr, ytr, Xva, yva, seed):
    tf.keras.utils.set_random_seed(seed)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy')
    cb = [tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)]
    t0 = time.perf_counter()
    h = model.fit(Xtr[:, None, :], ytr, validation_data=(Xva[:, None, :], yva), epochs=CFG['EPOCHS'],
                  batch_size=CFG['BATCH'], callbacks=cb, verbose=0)
    return time.perf_counter() - t0, len(h.history['loss'])

def predict_proba(model, X):
    return model.predict(X[:, None, :], batch_size=4096, verbose=0)

def time_model(model, X, batch, repeats, warmup):
    xb = tf.constant(X[:batch][:, None, :].astype('float32'))
    f = tf.function(lambda x: model(x, training=False))
    for _ in range(warmup):
        f(xb).numpy()
    ts = []
    for _ in range(repeats):
        t = time.perf_counter(); f(xb).numpy(); ts.append((time.perf_counter() - t) * 1000)
    return dict(batch=batch, median_ms=float(np.median(ts)), p95_ms=float(np.percentile(ts, 95)), mean_ms=float(np.mean(ts)))

In [ ]:
def run_one(pname, mname, seed, tr, te, feat, do_timing=False):
    classes = sorted(tr['label'].unique())
    cidx = {c: i for i, c in enumerate(classes)}
    ytr_all = tr['label'].map(cidx).values
    idx = np.arange(len(tr))
    strat = ytr_all if np.bincount(ytr_all).min() >= 2 else None
    i_tr, i_va = train_test_split(idx, test_size=CFG['VAL_FRAC'], stratify=strat, random_state=seed)
    pipe = Pipe().fit(tr.iloc[i_tr][feat].values)          # scaler/PCA fitted on training rows only
    Xtr = pipe.transform(tr.iloc[i_tr][feat].values); ytr = ytr_all[i_tr]
    Xva = pipe.transform(tr.iloc[i_va][feat].values); yva = ytr_all[i_va]
    Xte = pipe.transform(te[feat].values); yte_names = te['label'].values
    model = BUILDERS[mname](Xtr.shape[1], len(classes))
    train_s, n_ep = fit_model(model, Xtr, ytr, Xva, yva, seed)
    proba = predict_proba(model, Xte)
    res = evaluate(yte_names, proba, classes)
    res.update(protocol=pname, model=mname, seed=seed, train_seconds=train_s, epochs_run=n_ep,
               params=int(model.count_params()), n_train=int(len(i_tr)), n_features_in=int(Xtr.shape[1]))
    if do_timing:
        res['timing_ms'] = [time_model(model, Xte, b, CFG['TIMING_REPEATS'], CFG['TIMING_WARMUP'])
                            for b in CFG['TIMING_BATCHES']]
        raw_te = te[feat].values[:256]
        ts = []
        for _ in range(CFG['TIMING_REPEATS']):
            t = time.perf_counter(); pipe.transform(raw_te[:1]); ts.append((time.perf_counter() - t) * 1000)
        res['preprocess_ms_batch1_median'] = float(np.median(ts))
    return res, proba, classes, yte_names

In [ ]:
PROTOCOLS = {'daywise': (train_df, test_df)}
if CFG['RUN_RANDOM_POOLED_COMPARISON']:
    vc = df['label'].value_counts()
    pooled = df[df['label'].isin(vc[vc >= 2].index)].reset_index(drop=True)
    ptr, pte = train_test_split(pooled, test_size=0.30, stratify=pooled['label'], random_state=0)
    PROTOCOLS['random_pooled_70_30'] = (ptr.reset_index(drop=True), pte.reset_index(drop=True))

RESULTS, PROBS = [], {}
for pname, (tr, te) in PROTOCOLS.items():
    for mname in CFG['MODELS']:
        for seed in CFG['SEEDS']:
            first = (seed == CFG['SEEDS'][0]) and pname == 'daywise'
            res, proba, classes, yn = run_one(pname, mname, seed, tr, te, FEAT, do_timing=first)
            RESULTS.append(res)
            if seed == CFG['SEEDS'][0]:
                PROBS['%s_%s' % (pname, mname)] = dict(proba=proba.astype('float32'), y=yn, classes=np.array(classes))
            print(pname, mname, 'seed', seed, 'bin_acc %.4f bin_f1 %.4f mc_acc %.4f train_s %.0f'
                  % (res['bin_acc'], res['bin_f1'], res['mc_acc'], res['train_seconds']))

daywise hybrid seed 0 bin_acc 0.9943 bin_f1 0.9951 mc_acc 0.8447 train_s 205
daywise hybrid seed 1 bin_acc 0.9949 bin_f1 0.9956 mc_acc 0.8434 train_s 173
daywise hybrid seed 2 bin_acc 0.9937 bin_f1 0.9946 mc_acc 0.8409 train_s 74
daywise dnn_gru seed 0 bin_acc 0.9906 bin_f1 0.9919 mc_acc 0.8389 train_s 98
daywise dnn_gru seed 1 bin_acc 0.9896 bin_f1 0.9911 mc_acc 0.8376 train_s 88
daywise dnn_gru seed 2 bin_acc 0.9884 bin_f1 0.9901 mc_acc 0.8340 train_s 97
daywise blstm seed 0 bin_acc 0.9913 bin_f1 0.9925 mc_acc 0.8394 train_s 128
daywise blstm seed 1 bin_acc 0.9928 bin_f1 0.9938 mc_acc 0.8412 train_s 124
daywise blstm seed 2 bin_acc 0.9900 bin_f1 0.9914 mc_acc 0.8351 train_s 117
random_pooled_70_30 hybrid seed 0 bin_acc 0.9959 bin_f1 0.9965 mc_acc 0.9329 train_s 264
random_pooled_70_30 hybrid seed 1 bin_acc 0.9958 bin_f1 0.9964 mc_acc 0.9307 train_s 217
random_pooled_70_30 hybrid seed 2 bin_acc 0.9956 bin_f1 0.9962 mc_acc 0.9242 train_s 201
random_pooled_70_30 dnn_gru seed 0 bin_acc 0

In [ ]:
rows = []
for r in RESULTS:
    rows.append({k: v for k, v in r.items() if not isinstance(v, (dict, list))})
R = pd.DataFrame(rows)
keys = ['bin_acc', 'bin_precision', 'bin_recall', 'bin_f1', 'bin_fpr', 'bin_auc', 'bin_brier', 'bin_ece',
        'mc_acc', 'mc_macro_f1', 'mc_brier', 'mc_ece', 'train_seconds', 'params']
summary = R.groupby(['protocol', 'model'])[keys].agg(['mean', 'std'])
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 50)
print(summary.round(4).to_string())
print()
for r in RESULTS:
    if 'timing_ms' in r:
        print(r['protocol'], r['model'], r['timing_ms'], 'preprocess_ms(batch1)=%.3f' % r['preprocess_ms_batch1_median'])
print()
for r in RESULTS:
    if r['seed'] == CFG['SEEDS'][0] and r['unseen_family_detected_as_attack']:
        print(r['protocol'], r['model'], 'unseen-family detected as attack:', r['unseen_family_detected_as_attack'])

                            bin_acc         bin_precision         bin_recall          bin_f1         bin_fpr         bin_auc         bin_brier         bin_ece          mc_acc         mc_macro_f1         mc_brier          mc_ece         train_seconds              params     
                               mean     std          mean     std       mean     std    mean     std    mean     std    mean     std      mean     std    mean     std    mean     std        mean     std     mean     std    mean     std          mean      std       mean  std
protocol            model                                                                                                                                                                                                                                                         
daywise             blstm    0.9913  0.0014        0.9932  0.0025     0.9919  0.0003  0.9926  0.0012  0.0095  0.0035  0.9991  0.0002    0.0073  0.0005  0.0060  0.0012  0.8386 

In [ ]:
env = dict(tensorflow=tf.__version__, python=sys.version, platform=platform.platform(),
           gpu=sh('nvidia-smi --query-gpu=name,driver_version --format=csv,noheader'),
           cpu=sh('lscpu | grep "Model name"'), cfg=CFG, clean_log=CLEAN_LOG, features=FEAT,
           class_counts_train=train_df['label'].value_counts().to_dict(),
           class_counts_test=test_df['label'].value_counts().to_dict())
json.dump(env, open(CFG['OUT_DIR'] + '/environment.json', 'w'), indent=1, default=str)
json.dump(RESULTS, open(CFG['OUT_DIR'] + '/all_results.json', 'w'), indent=1, default=str)
summary.to_csv(CFG['OUT_DIR'] + '/summary.csv')
for k, v in PROBS.items():
    np.savez_compressed('%s/probs_%s.npz' % (CFG['OUT_DIR'], k), **v)
df[['_src', '_row', '_day', 'label']].to_csv(CFG['OUT_DIR'] + '/row_manifest.csv', index=False)
shutil.make_archive('/content/ddos_results', 'zip', CFG['OUT_DIR'])
from google.colab import files
files.download('/content/ddos_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>